# P1 — Factors Affecting Final Grades (student-mat)

**Main question:** How do studytime, absences, family support and midterm grades (G1/G2) relate to final grade (G3)?

- Dataset: UCI Student Performance (`data/raw/student-mat.csv`, 395 rows, `;`-separated) + Wikipedia PISA reference (`data/raw/pisa_reference.csv`)
- Pipeline: `StudentDataProcessor.load_data → clean_data → analyze → export_results` (see `data_processor.py`, run via `main.py`)
- Outputs: `data/processed/cleaned_students.csv`, `data/processed/summary.json`, 3 charts in `images/`

## 1. Python basics — 4 functions + data structures (rubric)

Required: `convert yes/no`, `pass/fail`, `absence buckets`, `average grade`; plus `list[dict]`, `dict` mapping, `set()` uniques, `tuple` thresholds.

In [ ]:
from data_processor import (
    THRESHOLDS, VAR_DESCRIPTIONS, SAMPLE_STUDENTS,
    convert_yes_no, calc_pass_fail, classify_absences, calc_average_grade,
)

# tuple threshold config
print('THRESHOLDS (PASS, G_WARN, ABS_MED, ABS_HIGH, FAIL_HIGH):', THRESHOLDS)
# dict mapping
print('studytime =', VAR_DESCRIPTIONS['studytime'])
# list of dicts
for s in SAMPLE_STUDENTS:
    print(s['id'], 'pass/fail:', calc_pass_fail(s['G3']), '| abs:', classify_absences(s['absences']),
          '| avg:', round(calc_average_grade(s['G1'], s['G2'], s['G3']), 2),
          '| internet_bin:', convert_yes_no(s['internet']))

## 2. Load + clean (OOP pipeline)

In [ ]:
from pathlib import Path
from data_processor import StudentDataProcessor

proc = StudentDataProcessor(
    Path('data/raw/student-mat.csv'), Path('data/raw/pisa_reference.csv'),
    Path('data/processed'), Path('images'))
df = proc.load_data()
print(df.shape, list(df.columns)[:6])
clean = proc.clean_data()
print(clean[['G1','G2','G3','pass_fail','absence_level','avg_grade','risk_group']].head(3).to_string())
print('unique schools:', set(clean['school'].unique()))
print('unique Mjob:', set(clean['Mjob'].unique()))

## 3. EDA — groupby mean/std (sub-questions)

Compare G3 by studytime, absences, failures, internet.

In [ ]:
res = proc.analyze()
import json; print(json.dumps({k: res[k] for k in ['n','pass_rate','n_pass','n_fail']}, indent=2))
for key in ['by_studytime','by_absence_level','by_failures','by_internet','by_risk_group']:
    print('\n##', key)
    for g, v in res[key].items():
        print(f"  {g}: mean={v['mean']} std={v['std']} n={v['n']}")

## 4. Charts (3 required)

Generated by `analyze()` into `images/`. Re-display here:

In [ ]:
from IPython.display import Image, display
display(Image('images/01_bar_studytime_internet.png'))
display(Image('images/02_hist_g3.png'))
display(Image('images/03_heatmap_corr.png'))

## 5. Correlation G1/G2/G3

In [ ]:
print(res['corr_G1_G2_G3'])
# Full matrix from summary.json
import pandas as pd
corr_full = pd.DataFrame(res['corr_full'])
print(corr_full.round(2).to_string())

## 6. PISA reference (web scraping)

`scrape_pisa_table()` fetched the Wikipedia ranking live with cache fallback.

In [ ]:
import pandas as pd
pisa = pd.read_csv('data/raw/pisa_reference.csv')
print(pisa.shape, list(pisa.columns))
print(pisa.head(5).to_string())

## 7. Insights + limitations

**Insights (from `summary.json`, n=395, pass rate 0.671):**

1. **G2 predicts G3 best** (r=0.905), then G1 (0.801) — midterm trajectory dominates demographics.
2. **Past failures hurt most:** failures=0 → mean 11.25 vs failures≥1 → 5.7–8.1 (gap ≈ 3–5.5 points, r≈−0.36).
3. **Study time helps modestly:** bucket 3 (5–10h) mean 11.4 vs bucket 1 (<2h) 10.05 (+1.35), but bucket 4 (n=27) stalls at 11.26 — diminishing returns, small-n caution.
4. **Internet gap:** yes 10.62 vs no 9.41 (+1.2), n_no=66 — real but confounded with SES.
5. **Absences paradox:** medium (4–9) mean 11.38 > high 10.37 > low 9.82; corr absences–G3 ≈ +0.03 ≈ zero. Absence count alone does not explain grades here — likely interacts with reason/ability.
6. **Risk groups separate cleanly:** low 13.63 (n=167) / medium 11.65 (n=91) / high 5.68 (n=137) — usable triage rule.

**Limitations:** single subject (mat), single school year, Portuguese cohort only; n=395 small for bucket 4; PISA table is external context only (different population/year); correlation ≠ causation; no train/test model — descriptive only.